# Inpaint

In [ ]:
import numpy as np
import torch
from PIL import Image
import os
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

# Give the path of your image
IMAGE_PATH= '/content/inpaint-example.png'
assert os.path.exists(IMAGE_PATH), f"Image path '{IMAGE_PATH}' does not exist."
# Read the image from the path
image= cv2.imread(IMAGE_PATH)
cv2_imshow(image)

In [ ]:
!pip install 'git+https://github.com/facebookresearch/segment-anything.git'
from segment_anything import sam_model_registry, SamPredictor

!pip install diffusers accelerate
from diffusers import StableDiffusionInpaintPipeline

!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
CHECKPOINT_PATH='/content/sam_vit_b_01ec64.pth'

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_b"

In [ ]:
# Convert to RGB format
image_rgb = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH)
sam.to(device=DEVICE)
mask_predictor = SamPredictor(sam)
mask_predictor.set_image(image_rgb)

# Provide points as input prompt [X, Y]-coordinates
input_point = np.array([[250, 250]])
input_label = np.array([1])

# Predicting Segmentation mask
masks, scores, logits = mask_predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=False,
)

mask = masks.astype(float) * 255
mask = np.transpose(mask, (1, 2, 0))
_ , bw_image = cv2.threshold(mask, 100, 255, cv2.THRESH_BINARY)
cv2_imshow(bw_image)
cv2.imwrite('mask.png', bw_image)

In [ ]:
# Load images using PIL
init_image = Image.open(IMAGE_PATH)
mask_image = Image.open('mask.png')

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
)
pipe = pipe.to(DEVICE)

prompt = "a grey cat sitting on a bench, face forward, high resolution"
inpaint_image = pipe(prompt=prompt, image=init_image, mask_image=mask_image).images[0]
inpaint_image.save('output.png')

In [ ]:
image = cv2.imread('/content/output.png')
cv2_imshow(image)

# Outpaint

In [ ]:
import numpy as np
import torch
from PIL import Image
import os
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

# load image and create an outpaint mask
image = cv2.imread('/content/inpaint-example.png')
image_rgb = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
height, width = image_rgb.shape[:2]
padding = 100   # num pixels to outpaint
mask = np.ones((height+2*padding, width+2*padding), dtype=np.uint8) * 255
mask[padding+2:-padding-2, padding+2:-padding-2] = 0
cv2_imshow(mask)
cv2.imwrite("mask2.png", mask)

In [ ]:
# extend the original image
image_extended = np.pad(image, ((padding, padding), (padding, padding), (0, 0)), mode='constant', constant_values=128)
cv2_imshow(image_extended)
cv2.imwrite("image_extended.png", image_extended)

In [ ]:
# Load images using PIL
init_image = Image.open('image_extended.png')
mask_image = Image.open('mask2.png')

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

inpaint_image = pipe(prompt="a dog on a bench in a park", image=init_image, mask_image=mask_image).images[0]
inpaint_image.save('output2.png')

In [ ]:
image = cv2.imread('/content/output2.png')
cv2_imshow(image)